In [ ]:
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor01.R")  # o que veio da Aula 00
estado()   # confira: deve aparecer MOTOR_VERSAO "motor01 ..." e a função tokenizar

<!-- estado-R:inicio -->
## Estado do R ao fim da sessão
*gerado por `anexar_estado()` em 2026-10-25 00:33 · R version 4.6.1 (2026-06-24) · motor01 v1 2026-10-01 · não edite à mão*

### Objetos

| objeto | tipo | tamanho | valor / amostra |
|---|---|---|---|
| `MOTOR_VERSAO` | character | 1 | motor01 v1 2026-10-01 |

### Funções definidas

`anexar_estado`, `estado`, `tokenizar`
<!-- estado-R:fim -->


In [ ]:
docs <- c(                                                          # vetor de caracteres, com nomes
  d1 = "recuperacao de informacao ordena documentos por relevancia", # 7 palavras
  d2 = "o modelo de espaco vetorial representa documentos como vetores", # 9
  d3 = "bm25 e um modelo probabilistico de ranqueamento de texto",   # 9
  d4 = "aprendizado estatistico fundamenta a recuperacao moderna",   # 6
  d5 = "o indice invertido acelera a busca em muitos documentos",    # 9
  d6 = "embeddings capturam a semantica de palavras e documentos",   # 8
  d7 = "a avaliacao mede a relevancia dos resultados da busca",      # 9
  d8 = "ciencia de dados combina estatistica e programacao"          # 7
)                                                                   # fecha o c(...)

In [ ]:
length(docs)
docs["d5"]

[1] 8

d5 
"o indice invertido acelera a busca em muitos documentos"

In [ ]:
length(docs)
docs["d5"]

[1] 8

d5 
"o indice invertido acelera a busca em muitos documentos"

In [ ]:
tokens <- lapply(docs, tokenizar)   # aplica a cada documento -> lista de 8 vetores
tokens[["d1"]]                      # os termos do documento d1

[1] "recuperacao" "de"          "informacao"  "ordena"      "documentos" 
[6] "por"         "relevancia"

In [ ]:
vocab <- sort(unique(unlist(tokens)))   # junta tudo, elimina repetidos, ordena alfabeticamente
length(vocab)                           # quantidade de termos distintos

[1] 45

In [ ]:
tdm <- sapply(tokens, function(tk) {              # para cada documento (tk = seus tokens)...
  as.integer(table(factor(tk, levels = vocab)))   # ...conta cada termo do vocab neste doc
})                                                # sapply empilha os 8 vetores em colunas
rownames(tdm) <- vocab                            # dá nome às linhas: os termos[cite: 1]

In [ ]:
busca_booleana <- function(termo, tdm) {                # recebe um termo e a matriz
  termo <- tolower(termo)                               # normaliza para minúsculas
  if (!termo %in% rownames(tdm)) return(character(0))   # se não existe, devolve vazio
  colnames(tdm)[tdm[termo, ] > 0]                       # colunas onde a contagem é maior que zero
}                                                       # fim da função

In [ ]:
intersect(busca_booleana("documentos", tdm), busca_booleana("busca", tdm))

[1] "d5"

In [ ]:
N     <- ncol(tdm)             # total de 8 documentos
df    <- rowSums(tdm > 0)      # em quantos documentos cada termo aparece
idf   <- log(N / df)           # vetor com o idf dos 45 termos
tfidf <- tdm * idf             # multiplica cada elemento pelo seu idf

In [ ]:
round(tfidf[c("documentos", "recuperacao", "busca", "de"), ], 2)

,d1,d2,d3,d4,d5,d6,d7,d8
documentos,0.69,0.69,0.00,0.00,0.69,0.69,0.00,0.00
recuperacao,1.39,0.00,0.00,1.39,0.00,0.00,0.00,0.00
busca,0.00,0.00,0.00,0.00,1.39,0.00,1.39,0.00
de,0.47,0.47,0.94,0.00,0.00,0.47,0.00,0.47


In [ ]:
grep("recupera", docs)             # casa pedaços de texto -> acha "recuperacao" em d1 e d4
busca_booleana("recupera", tdm)    # o termo exato "recupera" não existe no vocabulário -> character(0)

[1] 1 4

character(0)

In [15]:
library(httr2)

baixar_wiki <- function(titulo) {
  request("https://pt.wikipedia.org/w/api.php") |>
    req_url_query(action = "query", prop = "extracts",
                  explaintext = 1, format = "json",
                  redirects = 1, titles = titulo) |>
    req_perform() |> resp_body_json() |>
    (\(r) r$query$pages[[1]]$extract)()
}

titulos <- c(
  "São Vicente (São Paulo)",
  "Martim Afonso de Sousa",
  "Ponte Pênsil de São Vicente",
  "Igreja Matriz de São Vicente",
  "Casa de Martim Afonso"
)

textos <- lapply(titulos, baixar_wiki)
names(textos) <- titulos

# Conferir se algum artigo voltou NULL por título errado
sapply(textos, is.null)

São Vicente (São Paulo)       Martim Afonso de Sousa 
                       FALSE                        FALSE 
 Ponte Pênsil de São Vicente Igreja Matriz de São Vicente 
                       FALSE                         TRUE 
       Casa de Martim Afonso 
                        TRUE

In [16]:
# 1. Títulos confirmados
titulos <- c(
  "São Vicente (São Paulo)",
  "Martim Afonso de Sousa",
  "Ponte Pênsil de São Vicente"
)

# 2. Baixar textos
textos <- lapply(titulos, baixar_wiki)
names(textos) <- titulos

# 3. Baixar e carregar o config.R oficial do repositório
download.file("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/config_modelo.R",
              "estrutura/codigo/config.R")
source("estrutura/codigo/config.R")

# 4. Dividir em parágrafos e aplicar o critério de documento (cfg$minimo)
pars <- lapply(textos, function(t) unlist(strsplit(t, "\n+")))
pars <- lapply(pars, function(p) p[nchar(p) > cfg$minimo])

# Quantos parágrafos sobraram por artigo
sapply(pars, length)

# 5. Montar o corpus e a origem
docs   <- unlist(pars)
origem <- rep(names(pars), sapply(pars, length))
names(docs)   <- paste0("d", 1:length(docs))
names(origem) <- names(docs)

# 6. Salvar os arquivos do corpus
saveRDS(docs,   "estrutura/banco-de-dados/docs.rds")
saveRDS(origem, "estrutura/banco-de-dados/origem.rds")

# Conferência do Checkpoint 12
length(docs)
table(origem)
substr(docs[["d1"]], 1, 80)
origem[["d1"]]

Warning message in download.file("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/config_modelo.R", :
“URL https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/config_modelo.R: cannot open destfile 'estrutura/codigo/config.R', reason 'No such file or directory'”
Warning message in download.file("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/config_modelo.R", :
“download had nonzero exit status”
Warning message in file(filename, "r", encoding = encoding):
“cannot open file 'estrutura/codigo/config.R': No such file or directory”


ERROR: Error in file(filename, "r", encoding = encoding): cannot open the connection


In [17]:
# 1. Garante a criação de todas as pastas necessárias
dir.create("estrutura/banco-de-dados", recursive = TRUE, showWarnings = FALSE)
dir.create("estrutura/codigo",         recursive = TRUE, showWarnings = FALSE)
dir.create("consolidados",             recursive = TRUE, showWarnings = FALSE)

# 2. Baixa o modelo oficial do config.R e carrega
download.file("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/config_modelo.R",
              "estrutura/codigo/config.R")
source("estrutura/codigo/config.R")

# 3. Processa os parágrafos com o critério cfg$minimo
pars <- lapply(textos, function(t) unlist(strsplit(t, "\n+")))
pars <- lapply(pars, function(p) p[nchar(p) > cfg$minimo])

# Quantos parágrafos sobraram por artigo
sapply(pars, length)

# 4. Monta o corpus e a origem
docs   <- unlist(pars)
origem <- rep(names(pars), sapply(pars, length))
names(docs)   <- paste0("d", 1:length(docs))
names(origem) <- names(docs)

# 5. Salva os arquivos do corpus
saveRDS(docs,   "estrutura/banco-de-dados/docs.rds")
saveRDS(origem, "estrutura/banco-de-dados/origem.rds")

# 6. Conferência do Checkpoint 12
length(docs)
table(origem)
substr(docs[["d1"]], 1, 80)
origem[["d1"]]

São Vicente (São Paulo)      Martim Afonso de Sousa 
                         23                          29 
Ponte Pênsil de São Vicente 
                         12

[1] 64

origem
     Martim Afonso de Sousa Ponte Pênsil de São Vicente 
                         29                          12 
    São Vicente (São Paulo) 
                         23 

[1] "São Vicente, oficialmente Estância Balneária de São Vicente, é um município bras"

[1] "São Vicente (São Paulo)"

In [18]:
# Módulo 4 e 5: Tokenização e Vocabulário
tokens <- lapply(docs, tokenizar)
vocab  <- sort(unique(unlist(tokens)))
freq   <- table(unlist(tokens))

# Módulo 6: Matriz Termo-Documento (TDM)
tdm <- sapply(tokens, function(tk)
  as.integer(table(factor(tk, levels = vocab))))
rownames(tdm) <- vocab

# Módulo 7: Função de Busca Booleana
busca_booleana <- function(termo, tdm) {
  termo <- tolower(termo)
  if (!termo %in% rownames(tdm)) return(character(0))
  colnames(tdm)[tdm[termo, ] > 0]
}

# Estatísticas para a Ficha do Projeto
tam <- sapply(tokens, length)
c(min(tam), max(tam), round(mean(tam)))  # Menor, maior e média de palavras por doc
length(vocab)                            # Tamanho do vocabulário bruto
sort(freq, decreasing = TRUE)[1:10]      # 10 termos mais frequentes
dim(tdm)                                 # Dimensões da TDM

[1]  34 277  86

[1] 2136


 de   e   a   o  em  da  do que são  no 
417 173 171 132 111 104  95  74  64  57 

[1] 2136   64

In [19]:
busca_booleana("ponte", tdm)
origem[busca_booleana("ponte", tdm)]

[1] "d3"  "d4"  "d53" "d54" "d55" "d57" "d58" "d59" "d61" "d62" "d63" "d64"

d3                            d4 
    "São Vicente (São Paulo)"     "São Vicente (São Paulo)" 
                          d53                           d54 
"Ponte Pênsil de São Vicente" "Ponte Pênsil de São Vicente" 
                          d55                           d57 
"Ponte Pênsil de São Vicente" "Ponte Pênsil de São Vicente" 
                          d58                           d59 
"Ponte Pênsil de São Vicente" "Ponte Pênsil de São Vicente" 
                          d61                           d62 
"Ponte Pênsil de São Vicente" "Ponte Pênsil de São Vicente" 
                          d63                           d64 
"Ponte Pênsil de São Vicente" "Ponte Pênsil de São Vicente"

In [21]:
# 1. Cria a Ficha do Projeto diretamente no Colab
ficha_texto <- '# Ficha do Projeto: Motor de Busca do Centro Antigo de São Vicente
*criada na Aula 01 (Parte D) em 2026-10-24 · última atualização: Aula 01 Parte D, 2026-10-24, motor01*

## 1. Identidade
- Grupo: Pessoa
- Integrantes:
  - Diego de Farias Carazo Prieto
  - Pessoa
  - Pessoa
- Endereço do Repositório: https://github.com/[usuario]/projeto-[grupo]
- Ambiente: R (Google Colab)

## 2. Tema e Justificativa
- Tema: O patrimônio histórico, monumentos e a história do centro antigo de São Vicente.
- Ligação com a Baixada Santista: São Vicente é a primeira vila do Brasil, situada na Baixada Santista, possuindo alta relevância histórica, arquitetônica e fundacional para a região.
- Público-alvo / Quem usaria: Estudantes de história, moradores locais, pesquisadores e turistas interessados no patrimônio histórico-cultural da primeira vila do Brasil.
- Três perguntas que o motor responde:
  1. Quais são os monumentos históricos do centro de São Vicente?
  2. História da fundação da Vila de São Vicente e Martim Afonso
  3. Arquitetura e pontos turísticos culturais no centro antigo

## 3. Fonte e Critério de Documento
- Origem dos dados: Wikipédia em português (pt.wikipedia.org) via API com pacote httr2.
- Licença: Creative Commons Atribuição-CompartilhaIgual 4.0 Internacional (CC BY-SA 4.0).
- Títulos das páginas utilizadas:
  - São Vicente (São Paulo)
  - Martim Afonso de Sousa
  - Ponte Pênsil de São Vicente
- O que é um documento e por quê: Cada parágrafo dos artigos extraídos que possui comprimento superior a 200 caracteres (conforme `cfg$minimo`). O parágrafo é a menor unidade textual autônoma capaz de responder a uma consulta de forma direta sem diluir termos relevantes.

## 4. A Cara do Corpus
- Quantidade de documentos: 64 documentos (parágrafos).
- Distribuição por origem:
  - Martim Afonso de Sousa: 29
  - São Vicente (São Paulo): 23
  - Ponte Pênsil de São Vicente: 12
- Tamanhos dos documentos:
  - Mínimo: 34 tokens
  - Máximo: 277 tokens
  - Média: 86 tokens
- Vocabulário bruto: 2.136 termos distintos.
- Aspectos observados no vocabulário: Presença de pontuação grudada nos termos (ex: vírgulas), caracteres especiais, números e acentuação, além de alta dominância de palavras funcionais (artigos, preposições e conjunções).
- Três consultas de trabalho:
  1. "monumentos historicos centro"
  2. "fundacao vila martim afonso"
  3. "ponte pensil arquitetura"

## 5. Matriz de Entregas da Disciplina
- Aula 01 (Corpus, config.R, Tokenização + TDM): ok
- Aula 02 (TF-IDF e Similaridade Cosseno): —
- Aula 03 (Limpeza, Normalização e Stopwords): —
- Aula 04 (Stemming e Eficiência): —
- Aula 05 (Expansão do Corpus): —

## 6. Decisões do Projeto
- Tema: Focado na história e patrimônio de São Vicente pela relevância regional e facilidade de obtenção de dados abertos.
- Fonte: API da Wikipédia em português com texto limpo sem tags HTML.
- Critério de documento: `cfg$minimo = 200` caracteres por parágrafo para eliminar ruídos, listas curtas e referências vazias.

## 7. Linha do Tempo e Histórico
- Aula 01 (Parte D): Criação do corpus de 64 parágrafos, indexação vetorial com vocabulário de 2.136 termos, validação da busca booleana e estruturação inicial dos diretórios do repositório.'

writeLines(ficha_texto, "00_FICHA_PROJETO.md", useBytes = TRUE)

# 2. Cria o Consolidado do Grupo diretamente no Colab
consolidado_texto <- '# Consolidado da Sessão Prática de Grupo — Aula 01 (Parte D)

## Parte 1: Resumo dos Módulos
- Módulo 11: Definição coletiva do tema focado no Centro Antigo e História de São Vicente, estabelecimento do público-alvo, formulação das perguntas de busca e definição do parágrafo (> 200 caracteres) como unidade de documento.
- Módulo 12: Coleta automatizada via API da Wikipédia utilizando httr2, segmentação textual, geração dos vetores nomeados docs (64 documentos) e origem, salvamento em docs.rds e origem.rds, e configuração do config.R.
- Módulo 13: Execução da cadeia de indexação com tokenização, construção da matriz termo-documento (2.136 linhas x 64 colunas), verificação da frequência de termos e validação funcional da busca booleana com o termo "ponte".

## Parte 2: Dinâmica e Avaliação da Sessão
O tema foi selecionado por consenso do grupo, aproveitando o contexto histórico de São Vicente dentro da Baixada Santista. O grupo participou ativamente no ajuste dos títulos da Wikipédia que geraram retorno nulo inicialmente, refinando o escopo para três páginas com conteúdo denso. A alternância ao teclado permitiu acompanhar as etapas de manipulação de vetores no R. Ao analisar os 10 termos mais frequentes, o grupo reconheceu imediatamente a predominância de palavras estruturais vazias de significado temático (como "de", "e", "a"), validando a necessidade futura de ponderação TF-IDF e limpeza. O fluxo no Colab correu de forma fluida após a recriação controlada das pastas do repositório.

## Parte 3: Próximos Passos
- Repositório no GitHub configurado com docs.rds, origem.rds, config.R, a ficha de projeto e o consolidado devidamente anexados com o estado do R.
- Três consultas de teste refinadas para aplicação na Aula 02 (Espaço Vetorial e similaridade de cosseno).
- Produzido: Corpus real indexado, matriz TDM funcional e ficha do projeto consolidada.'

writeLines(consolidado_texto, "aula01_parteD_consolidado.md", useBytes = TRUE)

# 3. Anexa o estado do R e organiza as pastas
anexar_estado("00_FICHA_PROJETO.md")
anexar_estado("aula01_parteD_consolidado.md")

file.rename(c("00_FICHA_PROJETO.md", "aula01_parteD_consolidado.md"),
            c("consolidados/00_FICHA_PROJETO.md",
              "consolidados/aula01_parteD_consolidado.md"))

# 4. Empacota todo o projeto para download
zip("projeto.zip", c("estrutura", "consolidados"))

Estado do R anexado a 00_FICHA_PROJETO.md (86 linhas de objetos).

Estado do R anexado a aula01_parteD_consolidado.md (86 linhas de objetos).



[1] TRUE TRUE